# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abiodunadesesan/FlyRank-ml-Internship-/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
!pip -q install duckdb datasets huggingface_hub pyarrow pandas

from google.colab import userdata
from huggingface_hub import login

login(userdata.get("HF_TOKEN"))

from datasets import load_dataset
import pandas as pd
import duckdb

daily = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    streaming=True
)

rows = []

for i, row in enumerate(daily):
    rows.append(row)
    if i >= 50000:
        break

df = pd.DataFrame(rows)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Date range:", df["report_date"].min(), "to", df["report_date"].max())

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Rows: 50001
Columns: 30
Date range: 2025-01-27 to 2025-02-27


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## 1. Signal checks

### Signal 1: Search volume

I use `gsc_impressions` as the search-volume signal. This is linked to the FlyRank quick-win logic because impressions represent observed search demand.

I will compare CTR across impression-volume buckets.

The verdict will be based on the observed bucket table rather than assuming the relationship beforehand.

### Signal 2: CTR relative to search position

I use `gsc_clicks`, `gsc_impressions`, and `gsc_avg_position` to examine whether CTR differs across search-position buckets.

This is linked to the FlyRank CTR-fix logic.

The verdict will be based on the observed data.

### Baseline rule

Prioritize rows with meaningful search demand and below-median CTR within their observed search-position bucket.

Reason code: `CTR_OPPORTUNITY`

Action label: `REVIEW_CTR`


In [3]:
# Prepare fields used by the baseline.
signal_df = df.copy()

signal_df["gsc_impressions"] = pd.to_numeric(
    signal_df["gsc_impressions"], errors="coerce"
).fillna(0)

signal_df["gsc_clicks"] = pd.to_numeric(
    signal_df["gsc_clicks"], errors="coerce"
).fillna(0)

signal_df["gsc_avg_position"] = pd.to_numeric(
    signal_df["gsc_avg_position"], errors="coerce"
)

# CTR is only meaningful when impressions > 0.
signal_df["ctr"] = (
    signal_df["gsc_clicks"] /
    signal_df["gsc_impressions"].replace(0, pd.NA)
)

signal_df["ctr"] = pd.to_numeric(signal_df["ctr"], errors="coerce")

# Keep rows with usable GSC search data.
signal_df = signal_df[
    signal_df["gsc_data_available"].eq(True) &
    signal_df["client_has_gsc"].eq(True) &
    signal_df["gsc_impressions"].gt(0) &
    signal_df["gsc_avg_position"].notna()
].copy()

print("Rows with usable GSC data:", len(signal_df))

Rows with usable GSC data: 50000


In [4]:
# Bucket search volume using observed quartiles.
signal_df["impression_bucket"] = pd.qcut(
    signal_df["gsc_impressions"],
    q=4,
    labels=["LOW", "MEDIUM_LOW", "MEDIUM_HIGH", "HIGH"],
    duplicates="drop"
)

volume_check = (
    signal_df.groupby("impression_bucket", observed=True)
    .agg(
        n=("gsc_impressions", "size"),
        median_impressions=("gsc_impressions", "median"),
        median_ctr=("ctr", "median"),
        mean_ctr=("ctr", "mean")
    )
    .reset_index()
)

print(volume_check.to_string(index=False))

impression_bucket     n  median_impressions  median_ctr  mean_ctr
              LOW 14386                 2.0         0.0  0.008295
       MEDIUM_LOW 11866                 6.0         0.0  0.008628
      MEDIUM_HIGH 11260                12.0         0.0  0.007078
             HIGH 12488                31.0         0.0  0.006411


### Signal 2 verdict

I compare CTR across observed search-position buckets. The result is directional rather than a causal claim.

Verdict: [FILL AFTER RUNNING THE TABLE]


In [5]:
# Create search-position buckets.
signal_df["position_bucket"] = pd.cut(
    signal_df["gsc_avg_position"],
    bins=[0, 3, 10, 20, 100, float("inf")],
    labels=["TOP_3", "4_10", "11_20", "21_100", "100_PLUS"],
    right=True
)

position_check = (
    signal_df.groupby("position_bucket", observed=True)
    .agg(
        n=("gsc_avg_position", "size"),
        median_position=("gsc_avg_position", "median"),
        median_impressions=("gsc_impressions", "median"),
        median_ctr=("ctr", "median"),
        mean_ctr=("ctr", "mean")
    )
    .reset_index()
)

print(position_check.to_string(index=False))

position_bucket     n  median_position  median_impressions  median_ctr  mean_ctr
          TOP_3   932         2.500000                 3.0         0.0  0.038792
           4_10 13540         7.250000                 6.0         0.0  0.013211
          11_20  9324        14.333333                11.0         0.0  0.008950
         21_100 26074        41.800000                 8.0         0.0  0.002780
       100_PLUS    31       101.000000                 1.0         0.0  0.000000


## 2. Build the ranked queue

The baseline scores each eligible row using two observed signals:

1. Search demand, measured by impressions.
2. CTR weakness relative to the median CTR of the same search-position bucket.

A higher score means more observed opportunity for a CTR review.

The rule uses only information available on the current row. It does not use future performance or a target label.

Reason code: `CTR_OPPORTUNITY`

Action: `REVIEW_CTR`


In [6]:
# Work from the signal table created above.
queue = signal_df.copy()

# Calculate the median CTR for each position bucket.
position_ctr_median = (
    queue.groupby("position_bucket", observed=True)["ctr"]
    .median()
    .rename("position_bucket_median_ctr")
)

queue = queue.join(
    position_ctr_median,
    on="position_bucket"
)

# CTR gap:
# positive = the row is below the typical CTR for its position bucket.
queue["ctr_gap"] = (
    queue["position_bucket_median_ctr"] - queue["ctr"]
).clip(lower=0)

# Convert impressions into a percentile-style volume score.
queue["volume_score"] = queue["gsc_impressions"].rank(
    pct=True,
    method="average"
)

# Convert CTR weakness into a percentile-style score.
queue["ctr_gap_score"] = queue["ctr_gap"].rank(
    pct=True,
    method="average"
)

# One baseline score.
queue["score"] = (
    0.5 * queue["volume_score"] +
    0.5 * queue["ctr_gap_score"]
)

# One reason code only.
queue["reason_code"] = "CTR_OPPORTUNITY"

# Action label.
queue["action"] = "REVIEW_CTR"

# Rank highest opportunity first.
queue = queue.sort_values(
    ["score", "gsc_impressions"],
    ascending=[False, False]
).reset_index(drop=True)

queue["rank"] = queue.index + 1

# Select the fields needed for the ranked queue.
baseline_queue = queue[
    [
        "rank",
        "report_date",
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ctr",
        "position_bucket",
        "position_bucket_median_ctr",
        "ctr_gap",
        "score",
        "reason_code",
        "action"
    ]
].copy()

print(baseline_queue.head(10).to_string(index=False))

 rank report_date          client_hash_id          content_hash_id  gsc_impressions  gsc_clicks  gsc_avg_position      ctr position_bucket  position_bucket_median_ctr  ctr_gap    score     reason_code     action
    1  2025-02-24 client_73cda7b4e4f265ea content_690b092cf66bc2a4              818          13          1.553790 0.015892           TOP_3                         0.0      0.0 0.750005 CTR_OPPORTUNITY REVIEW_CTR
    2  2025-02-25 client_73cda7b4e4f265ea content_690b092cf66bc2a4              742          16          1.849057 0.021563           TOP_3                         0.0      0.0 0.749995 CTR_OPPORTUNITY REVIEW_CTR
    3  2025-02-26 client_73cda7b4e4f265ea content_690b092cf66bc2a4              714          13          1.652661 0.018207           TOP_3                         0.0      0.0 0.749985 CTR_OPPORTUNITY REVIEW_CTR
    4  2025-02-23 client_73cda7b4e4f265ea content_690b092cf66bc2a4              590           8          1.833898 0.013559           TOP_3              

In [7]:
from pathlib import Path

output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

baseline_queue.to_csv(output_path, index=False)

print(f"Wrote {len(baseline_queue):,} rows to {output_path}")

Wrote 50,000 rows to work/outputs/baseline_action_score.csv


## 3. Top-20 review

The following review examines the highest-ranked rows from the baseline queue.

For each row, I record the action, reason code, confidence note, and what would make the recommendation wrong.

These are decision-support recommendations, not claims that the pages will definitely improve.


In [8]:
top20 = baseline_queue.head(20).copy()

top20["confidence_note"] = top20.apply(
    lambda row: (
        f"Observed {row['gsc_impressions']:.0f} impressions with "
        f"CTR {row['ctr']:.4f}, below the position-bucket median "
        f"of {row['position_bucket_median_ctr']:.4f}."
    ),
    axis=1
)

top20["what_would_make_it_wrong"] = top20.apply(
    lambda row: (
        "The recommendation could be wrong if the low CTR reflects "
        "search intent, SERP features, brand demand, or measurement "
        "issues rather than a page-level CTR opportunity."
    ),
    axis=1
)

review_columns = [
    "rank",
    "action",
    "reason_code",
    "confidence_note",
    "what_would_make_it_wrong"
]

display(top20[review_columns])

,rank,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,REVIEW_CTR,CTR_OPPORTUNITY,"Observed 818 impressions with CTR 0.0159, belo...",The recommendation could be wrong if the low C...
1,2,REVIEW_CTR,CTR_OPPORTUNITY,"Observed 742 impressions with CTR 0.0216, belo...",The recommendation could be wrong if the low C...
2,3,REVIEW_CTR,CTR_OPPORTUNITY,"Observed 714 impressions with CTR 0.0182, belo...",The recommendation could be wrong if the low C...
3,4,REVIEW_CTR,CTR_OPPORTUNITY,"Observed 590 impressions with CTR 0.0136, belo...",The recommendation could be wrong if the low C...
4,5,REVIEW_CTR,CTR_OPPORTUNITY,"Observed 580 impressions with CTR 0.0138, belo...",The recommendation could be wrong if the low C...
5,6,REVIEW_CTR,CTR_OPPORTUNITY,"Observed 535 impressions with CTR 0.0093, belo...",The recommendation could be wrong if the low C...
6,7,REVIEW_CTR,CTR_OPPORTUNITY,"Observed 534 impressions with CTR 0.0281, belo...",The recommendation could be wrong if the low C...
7,8,REVIEW_CTR,CTR_OPPORTUNITY,"Observed 532 impressions with CTR 0.0207, belo...",The recommendation could be wrong if the low C...
8,9,REVIEW_CTR,CTR_OPPORTUNITY,"Observed 506 impressions with CTR 0.0217, belo...",The recommendation could be wrong if the low C...
9,10,REVIEW_CTR,CTR_OPPORTUNITY,"Observed 503 impressions with CTR 0.0159, belo...",The recommendation could be wrong if the low C...


## 4. Weak picks + leakage check

Some high-ranked rows may still be weak recommendations.

A high score does not prove that changing the page will increase clicks. The rule can be wrong when CTR is affected by search intent, SERP features, brand/non-brand differences, seasonality, or measurement quality.

The baseline uses current-row search metrics only. It does not use a future window, a target label, or a product flag as an input.


In [9]:
# Show possible weak picks among the top 20.
weak_picks = top20[
    (top20["gsc_impressions"] < top20["gsc_impressions"].median()) |
    (top20["ctr_gap"] == 0)
].copy()

print("Potential weak picks:", len(weak_picks))

if len(weak_picks) > 0:
    display(
        weak_picks[
            [
                "rank",
                "gsc_impressions",
                "gsc_avg_position",
                "ctr",
                "position_bucket_median_ctr",
                "ctr_gap",
                "score"
            ]
        ]
    )
else:
    print("No obvious weak picks found using these checks.")

Potential weak picks: 20


,rank,gsc_impressions,gsc_avg_position,ctr,position_bucket_median_ctr,ctr_gap,score
0,1,818,1.553790,0.015892,0.0,0.0,0.750005
1,2,742,1.849057,0.021563,0.0,0.0,0.749995
2,3,714,1.652661,0.018207,0.0,0.0,0.749985
3,4,590,1.833898,0.013559,0.0,0.0,0.749975
4,5,580,4.984483,0.013793,0.0,0.0,0.749965
5,6,535,23.874766,0.009346,0.0,0.0,0.749955
6,7,534,1.750936,0.028090,0.0,0.0,0.749945
7,8,532,1.832707,0.020677,0.0,0.0,0.749935
8,9,506,7.045455,0.021739,0.0,0.0,0.749925
9,10,503,2.880716,0.015905,0.0,0.0,0.749910


In [10]:
# Leakage check.
future_columns = [
    "future",
    "next",
    "label",
    "target",
    "outcome",
    "conversion"
]

used_columns = set(queue.columns)

possible_leakage = [
    col for col in queue.columns
    if any(term in col.lower() for term in future_columns)
]

print("Possible future/label-derived columns:", possible_leakage)

assert len(possible_leakage) == 0, (
    f"Potential leakage columns found: {possible_leakage}"
)

print("Leakage check passed.")

Possible future/label-derived columns: []
Leakage check passed.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.